In [ ]:
!pip install datasets transformers pennylane pennylane-lightning-gpu --upgrade
!pip install custatevec-cu12 cuquantum-cu12

# CONFIG


In [ ]:
import torch

class Config:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_samples = 500
    test_samples = 200
    batch_size = 4

    n_qubits = 6
    K = 2
    T = 2
    n_layers = 4

    epochs = 10
    lr = 1e-3   # 🔥 increased

cfg = Config()

# DATA PIPELINE


In [ ]:
import re
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

dataset = load_dataset("gsm8k", "main")

train_data = [dataset["train"][i] for i in range(cfg.train_samples)]
test_data = [dataset["test"][i] for i in range(cfg.test_samples)]

def extract_label(example):
    match = re.search(r"#### (\d+)", example["answer"])
    if match:
        return min(int(match.group(1)) // 10, 99)
    return 0

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
encoder = AutoModel.from_pretrained("distilbert-base-uncased").to(cfg.device)
encoder.eval()

def encode_batch(texts):
    inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt").to(cfg.device)
    with torch.no_grad():
        outputs = encoder(**inputs)
    return outputs.last_hidden_state[:, 0, :]

def build_encoded_dataset(data):
    encoded = []
    for i in tqdm(range(0, len(data), 32), desc="Encoding"):
        batch = data[i:i+32]
        texts = [x["question"] for x in batch]
        labels = [extract_label(x) for x in batch]

        emb = encode_batch(texts).cpu()

        for j in range(len(batch)):
            encoded.append((emb[j], labels[j]))
    return encoded

train_encoded = build_encoded_dataset(train_data)
test_encoded = build_encoded_dataset(test_data)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Encoding: 100%|██████████| 7/7 [00:00<00:00,  7.70it/s]


# QUANTUM NODE


In [ ]:
import pennylane as qml
import torch.nn as nn
import math

try:
    dev = qml.device("lightning.gpu", wires=cfg.n_qubits)
    print("⚡ Using lightning.gpu")
except:
    dev = qml.device("default.qubit", wires=cfg.n_qubits)
    print("⚠️ Using default.qubit")

@qml.qnode(dev, interface="torch", diff_method="adjoint")
def qnode_batch(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(cfg.n_qubits))
    qml.StronglyEntanglingLayers(weights, wires=range(cfg.n_qubits))
    return tuple(qml.expval(qml.PauliZ(i)) for i in range(cfg.n_qubits))

⚡ Using lightning.gpu


# CLASSICAL BASELINE


In [ ]:
class ClassicalModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Linear(256, 100)
        )

    def forward(self, x):
        return self.net(x)

# qAIR v10 MODEL


In [ ]:
class QuantumV11(nn.Module):
    def __init__(self):
        super().__init__()

        self.K = cfg.K
        self.T = cfg.T

        self.proj = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Linear(256, cfg.n_qubits)
        )

        self.step_embed = nn.Embedding(cfg.T, cfg.n_qubits)

        self.q_weights = nn.Parameter(
            0.01 * torch.randn(cfg.n_layers, cfg.n_qubits, 3)
        )

        self.refine = nn.Sequential(
            nn.Linear(cfg.n_qubits, cfg.n_qubits),
            nn.Tanh()
        )

        self.score = nn.Sequential(
            nn.Linear(cfg.n_qubits, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

        self.decoder = nn.Sequential(
            nn.Linear(cfg.n_qubits, 128),
            nn.ReLU(),
            nn.Linear(128, 100)
        )

    def forward(self, x, return_all=False):

        B = x.shape[0]

        base = torch.tanh(self.proj(x)) * math.pi

        # 🔥 STRONG DIVERSITY INITIALIZATION
        state = torch.stack([
            base * (1 + 0.3 * i) + 0.5 * torch.randn_like(base)
            for i in range(self.K)
        ], dim=1)

        all_states = []

        for t in range(self.T):

            step_emb = self.step_embed(torch.tensor(t, device=x.device))
            state = state + step_emb

            # 🔥 REPULSIVE ENTANGLEMENT (CRITICAL)
            diff = state.unsqueeze(2) - state.unsqueeze(1)
            repulsion = diff.mean(dim=2)
            state = state + 0.3 * repulsion

            # 🔥 QUANTUM ONLY AT FIRST STEP
            if t == 0:
                s_all = state.view(-1, cfg.n_qubits)

                q_out = qnode_batch(s_all, self.q_weights)
                q_out = torch.stack(q_out, dim=-1)

                q_out = q_out.to(x.device).float()

                # 🔥 NORMALIZATION (STABILITY)
                q_out = q_out / (torch.norm(q_out, dim=-1, keepdim=True) + 1e-6)

                q_out = q_out.view(B, self.K, cfg.n_qubits)

                state = state + 0.5 * self.refine(q_out)

            else:
                state = state + 0.3 * self.refine(state)

            state = torch.clamp(state, -math.pi, math.pi)

            all_states.append(state)

        # 🔥 STRONGER COLLAPSE
        scores = self.score(state).squeeze(-1)
        scores = scores + torch.norm(state, dim=-1)

        weights = torch.softmax(scores, dim=1)

        collapsed = (state * weights.unsqueeze(-1)).sum(dim=1)

        logits = self.decoder(collapsed)

        if return_all:
            return logits, state, all_states, weights

        return logits

# TRAIN LOOP


In [ ]:
from torch.cuda.amp import autocast, GradScaler

def train(model, data):

    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)
    scaler = GradScaler()
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(cfg.epochs):

        model.train()
        total_loss = 0
        correct = 0
        total = 0

        for xb, yb in build_loader(data):

            with autocast():
                logits, state, _, _ = model(xb, return_all=True)

                ce_loss = loss_fn(logits, yb)

                # 🔥 DIVERSITY LOSS
                div_loss = -torch.mean(torch.var(state, dim=1))

                loss = ce_loss + 0.1 * div_loss

            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item()

            preds = logits.argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += len(yb)

        print(f"Epoch {epoch+1}: Loss={total_loss:.4f}, Acc={(correct/total)*100:.2f}%")

# EVALUATION


In [ ]:
import time
from sklearn.metrics import confusion_matrix

def evaluate(model, data):

    model.eval()

    correct = 0
    total = 0
    best_of_k = 0

    all_preds = []
    all_labels = []

    is_quantum = hasattr(model, "K")

    start = time.time()

    with torch.no_grad():
        for xb, yb in build_loader(data):

            if is_quantum:
                logits, state, _, _ = model(xb, return_all=True)
            else:
                logits = model(xb)
                state = None

            preds = logits.argmax(dim=1)
            correct += (preds == yb).sum().item()

            if is_quantum:
                for k in range(model.K):
                    logits_k = model.decoder(state[:, k, :])
                    preds_k = logits_k.argmax(dim=1)
                    best_of_k += (preds_k == yb).sum().item()

            total += len(yb)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(yb.cpu().numpy())

    end = time.time()

    acc = correct / total
    best_k_acc = best_of_k / (total * model.K) if is_quantum else None
    tps = (end - start) / total

    cm = confusion_matrix(all_labels, all_preds)

    print("\n📊 Evaluation")
    print(f"Accuracy: {acc*100:.2f}%")
    if best_k_acc is not None:
        print(f"Best-of-K: {best_k_acc*100:.2f}%")
    print(f"Time/sample: {tps:.4f}s")

    return acc, best_k_acc, cm

# RUN


In [ ]:
models = {
    "Classical": ClassicalModel().to(cfg.device),
    "Quantum v10.5": QuantumV105().to(cfg.device)
}

results = {}

for name, model in models.items():

    print(f"\n🚀 Training {name}")
    train(model, train_encoded)

    acc, best_k, cm = evaluate(model, test_encoded)

    results[name] = acc

print("\n📊 FINAL RESULTS")
for k, v in results.items():
    print(f"{k}: {v*100:.2f}%")


🚀 Training Classical


/tmp/ipykernel_6591/958157106.py:13: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_6591/958157106.py:25: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 1: Loss=435.6616, Acc=18.00%
Epoch 2: Loss=380.4231, Acc=18.80%
Epoch 3: Loss=372.9739, Acc=19.60%
Epoch 4: Loss=366.7056, Acc=21.20%
Epoch 5: Loss=360.7661, Acc=23.20%
Epoch 6: Loss=354.8232, Acc=24.60%
Epoch 7: Loss=348.7172, Acc=25.20%
Epoch 8: Loss=342.6501, Acc=25.60%
Epoch 9: Loss=336.2831, Acc=26.20%
Epoch 10: Loss=330.2396, Acc=27.40%

📊 Evaluation
Accuracy: 18.50%
Time/sample: 0.0001s

🚀 Training Quantum v10.5


/tmp/ipykernel_6591/958157106.py:13: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_6591/958157106.py:25: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 1: Loss=463.2417, Acc=11.60%
Epoch 2: Loss=387.8647, Acc=19.40%
Epoch 3: Loss=383.1421, Acc=19.40%
Epoch 4: Loss=381.7478, Acc=19.40%
Epoch 5: Loss=380.8665, Acc=19.40%
Epoch 6: Loss=380.2866, Acc=19.40%
Epoch 7: Loss=379.6711, Acc=19.40%
Epoch 8: Loss=379.6084, Acc=19.40%
Epoch 9: Loss=379.2041, Acc=19.40%
Epoch 10: Loss=377.5869, Acc=19.40%

📊 Evaluation
Accuracy: 15.00%
Best-of-K: 15.00%
Time/sample: 0.0939s

📊 FINAL RESULTS
Classical: 18.50%
Quantum v10.5: 15.00%
